In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard imports
import numpy as np
import xarray as xr
import tqdm as tqdm

# For variogram estimation
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import label
from scipy.spatial.distance import pdist
from scipy.optimize import curve_fit

In [3]:
# Import OpenSense modules as submodules
import sys
import os

sys.path.append(os.path.abspath("./pycomlink/"))
sys.path.append(os.path.abspath("./poligrain/src/"))
sys.path.append(os.path.abspath("./mergeplg/src/"))

import pycomlink as pycml 
import poligrain as plg
import mergeplg 

In [4]:
# Define function to estimate variogram from rainfall event
def get_event_variogram(da_event, bin_edges, min_obs, plot_variogram=False):
    """
    da_event: xarray data for computing variogram
    bin_edges: array of distance bins [m] (e.g., np.linspace(0, 30000, 1500))
    min_obs: minimum observations needed to perform variogram estimation
    plot_variogram: whether to plot the variogram
    """
    all_distances = []
    all_sq_diffs = []
    
    # 1. Collect pairs from all time steps in the event
    n_obs = 0
    for t in da_event.time:
        # Extract data for this timestamp and drop nan
        data_t = da_event.sel(time=t).dropna(dim='cml_id')
        
        # We need at least 2 points to make a pair
        if len(data_t.cml_id) < 2:
            continue

        n_obs += data_t.cml_id.size

        # Get coordinates of data
        coords = np.column_stack([data_t.x, data_t.y])
        values = data_t.values
        
        # Calculate distances and 0.5 * (zi - zj)^2
        dist = pdist(coords, metric='euclidean')

        # Calculate squared distance (variance)
        sq_diff = 0.5 * pdist(values[:, None], 'sqeuclidean')
        
        all_distances.append(dist)
        all_sq_diffs.append(sq_diff)

    # If not enough observations
    if n_obs < min_obs:
        return None
        
    # Flatten into two long arrays of all pairs in the event
    dist_pool = np.concatenate(all_distances)
    diff_pool = np.concatenate(all_sq_diffs)
    
    # 2. Binning 
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    gamma_obs = []
    count = []

    # Also drop largest outliers
    #diff_upper = np.nanquantile(dist_pool, q = 0.95)
    
    for i in range(len(bin_edges)-1):
        mask = (dist_pool > bin_edges[i]) & (dist_pool <= bin_edges[i+1]) # & (diff_pool <= diff_upper) 
        if np.any(mask):
            gamma_obs.append(np.mean(diff_pool[mask]))
            count.append(diff_pool[mask].size)
        else:
            gamma_obs.append(np.nan)
            count.append(0)

    count = np.array(count)
    gamma_obs = np.array(gamma_obs)

    # 3. Define valid observation bins and get variance
    valid = ~np.isnan(gamma_obs) & (count != 0) 

    # Locate where most observations are
    dist_lower = np.nanquantile(dist_pool, q = 0.05)
    dist_upper = np.nanquantile(dist_pool, q = 0.95)
    ind_lower = np.where(dist_lower < bin_centers[valid])[0][0]

    if (dist_upper > bin_centers[valid]).all():
        ind_upper = bin_centers[valid].size
    else:
        ind_upper = np.where(dist_upper < bin_centers[valid])[0][0]

    total_variance = np.nanmean(gamma_obs[valid][ind_lower:ind_upper])
    gamma_obs_norm = gamma_obs/total_variance

    # If no variance 
    if total_variance == 0: 
        return None
        
    # 4. Define variogram and bounds
    def spherical_model(h, nugget, p_sill, range_a):
        # Use same definitions as pykrige:
        # https://geostat-framework.readthedocs.io/projects/pykrige/en/stable/variogram_models.html
        return np.where(h <= range_a, 
                        p_sill * (1.5 * (h/range_a) - 0.5 * (h/range_a)**3) + nugget, 
                        p_sill + nugget)
        
    # Estimate partial sill from normalized variance
    def f(h, n, r):
        return spherical_model(h, n, 1 - n, r)
        
    # Initial guess: [min(gamma), max_dist/2]
    p0 = [0, np.max(bin_centers[valid][ind_lower:ind_upper])/2]

    # Parameter bounds
    bound_l_nugget = 0
    bound_l_range = bin_centers[valid][ind_lower]
    bound_u_nugget = np.min([np.nanmean(gamma_obs_norm[valid][:ind_lower+1]), 1])
    bound_u_range = np.max(bin_edges)*2

    if bound_u_nugget == 0:
        bound_u_nugget = 0.01 # For numerics
    
    bounds = [[bound_l_nugget, bound_l_range], [bound_u_nugget, bound_u_range]]
    
    # 5. Optimize and return parameters
    popt, _ = curve_fit(
        f, 
        bin_centers[valid][:ind_upper], 
        gamma_obs_norm[valid][:ind_upper], 
        p0=p0, 
        bounds=bounds,
    )

    nugget = popt[0]
    p_sill = 1 - nugget
    range_a = popt[1]

    if plot_variogram:
        fig, ax = plt.subplots(1, 1)
        ax.plot(bin_centers[valid], spherical_model(bin_centers[valid], nugget, p_sill, range_a), label='plot variogram')
        ax.plot(bin_centers[valid][:ind_upper], gamma_obs_norm[valid][:ind_upper], label='data used')
        ax.plot(bin_centers[valid][ind_upper:], gamma_obs_norm[valid][ind_upper:], label='data left out')
        plt.legend()
        plt.show()
    
    return n_obs, [nugget, p_sill, range_a]


In [5]:
# Create folder for storing files
os.makedirs('data/adjusted_fields', exist_ok=True)

In [6]:
# Number of neighbors
nnear = 12 

# OpenMRG - Adjust rainfall fields

In [7]:
# OpenMRG
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")                    
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc")       

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)

In [8]:
# additive POINT BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_add_pBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_add_pBK_adpt_small.nc')

del data, merger

100%|██████████| 168/168 [00:12<00:00, 13.11it/s]


In [9]:
# multiplicative POINT BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_mul_pBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_mul_pBK_adpt_small.nc')

del data, merger

 90%|████████▉ | 26/29 [00:13<00:01,  2.65it/s]/tmp/ipykernel_474337/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|██████████| 168/168 [00:08<00:00, 19.19it/s]


In [10]:
# additive LINE BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_add_lBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_add_lBK_adpt_small.nc')

del data, merger

100%|██████████| 168/168 [00:10<00:00, 16.37it/s]


In [11]:
# multiplicative LINE BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_mul_lBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_mul_lBK_adpt_small.nc')

del data, merger

 90%|████████▉ | 26/29 [00:11<00:00,  3.04it/s]/tmp/ipykernel_474337/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|██████████| 168/168 [00:08<00:00, 19.99it/s]


In [12]:
# KED point

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_pked_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_pked_adpt_small.nc')

del data, merger

 90%|████████▉ | 26/29 [00:11<00:01,  2.84it/s]/tmp/ipykernel_474337/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|██████████| 168/168 [00:10<00:00, 15.89it/s]


In [13]:
# KED line

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_lked_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_rainfall_lked_adpt_small.nc')

del data, merger

 90%|████████▉ | 26/29 [00:11<00:01,  2.97it/s]/tmp/ipykernel_474337/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|██████████| 168/168 [00:10<00:00, 16.12it/s]


# OpenRainER

In [14]:
# OpenRainER
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")         
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc")   
ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)

In [15]:
# additive POINT BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_add_pBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_add_pBK_adpt_small.nc')

del data, merger

100%|██████████| 264/264 [02:21<00:00,  1.86it/s]


In [16]:
# multiplicative POINT BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_mul_pBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_mul_pBK_adpt_small.nc')

del data, merger

100%|██████████| 264/264 [01:28<00:00,  2.98it/s]


In [17]:
# additive LINE BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_add_lBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_add_lBK_adpt_small.nc')

del data, merger

100%|██████████| 264/264 [02:23<00:00,  1.84it/s]


In [18]:
# multiplicative LINE BLOCK KRIGING

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':(0.1,15)},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_mul_lBK_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_mul_lBK_adpt_small.nc')

del data, merger

100%|██████████| 264/264 [01:28<00:00,  2.97it/s]


In [19]:
# KED point

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_pked_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_pked_adpt_small.nc')

del data, merger

100%|██████████| 264/264 [02:19<00:00,  1.89it/s]


In [20]:
# KED line

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':10},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"rainfall_lked_adpt_small": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_rainfall_lked_adpt_small.nc')

del data, merger

  0%|          | 0/29 [00:00<?, ?it/s]

100%|██████████| 264/264 [02:19<00:00,  1.89it/s]
